In [2]:
import pandas as pd
import sqlite3
import logging
import os

# -----------------------------
# Logging Setup
# -----------------------------
if not os.path.exists('logs'):
    os.makedirs('logs')

logging.basicConfig(
    filename='logs/pipeline.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# -----------------------------
# Task 1: Ingest Data
# -----------------------------
def ingest_data():
    logging.info("Starting ingestion...")
    
    df = pd.read_csv('raw_data/faculty_workload_batch.csv')
    df.to_csv('processed_data/raw_faculty.csv', index=False)
    
    logging.info("Ingestion completed.")

# -----------------------------
# Task 2: Clean Data
# -----------------------------
def clean_data():
    logging.info("Starting cleaning...")
    
    df = pd.read_csv('processed_data/raw_faculty.csv')
    df = df.dropna()
    
    df.to_csv('processed_data/clean_faculty.csv', index=False)
    
    logging.info("Cleaning completed.")

# -----------------------------
# Task 3: Validate Data
# -----------------------------
def validate_data():
    logging.info("Starting validation...")
    
    df = pd.read_csv('processed_data/clean_faculty.csv')

    if not df['faculty_id'].notnull().all():
        raise Exception("Null faculty_id found!")

    if df['hours'].max() > 40:
        raise Exception("Workload exceeds limit!")

    logging.info("Validation successful.")

# -----------------------------
# Task 4: Load to Database
# -----------------------------
def load_data():
    logging.info("Loading to database...")
    
    df = pd.read_csv('processed_data/clean_faculty.csv')
    
    conn = sqlite3.connect('processed_data/faculty.db')
    df.to_sql('faculty_workload', conn, if_exists='replace', index=False)
    conn.close()
    
    logging.info("Data loaded successfully.")

# -----------------------------
# Orchestration Function
# -----------------------------
def run_pipeline():
    logging.info("Pipeline started")

    try:
        ingest_data()
        clean_data()
        validate_data()
        load_data()
        logging.info("Pipeline completed successfully")

    except Exception as e:
        logging.error(f"Pipeline failed: {e}")

# -----------------------------
# Entry Point
# -----------------------------
if __name__ == "__main__":
    run_pipeline()